# MASFE real MODIS policy replay

This notebook replays the cached Westlands MODIS run generated by `real_modis_replay.py`. It is presentation-oriented: all network work should be done before opening the notebook.

In [ ]:
from pathlib import Path
import json

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

root = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
runs = sorted((root / 'outputs' / 'real_modis').glob('*'))
if not runs:
    raise FileNotFoundError('No cached real_modis outputs found. Run real_modis_replay.py first.')
run_dir = runs[-1]
metrics = json.loads((run_dir / 'replay_metrics.json').read_text())
timeline = pd.read_csv(run_dir / 'action_timeline.csv', parse_dates=['date'])
stack = np.load(run_dir / 'csc_stack.npz')
print('Using run:', run_dir)
metrics

In [ ]:
ACTION_COLORS = {
    'BASELINE': '#bdbdbd',
    'MOD13': '#1f77b4',
    'FUSE': '#ff7f0e',
    'FUSE_PRIORITY': '#d62728',
    'SKIP': '#7f7f7f',
}

def render_step(index: int):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    csc_map = stack['csc'][index]
    im = axes[0].imshow(csc_map, cmap='inferno', vmin=0.0, vmax=max(0.7, float(np.nanmax(stack['csc']))))
    axes[0].set_title(f"{timeline.loc[index, 'date'].date()} | {timeline.loc[index, 'action']}")
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label='CSC')

    axes[1].plot(timeline['date'], timeline['csc_max'], color='#d62728', linewidth=2, label='CSC max')
    axes[1].plot(timeline['date'], timeline['valid_fraction'], color='#1f77b4', linewidth=2, label='Valid fraction')
    axes[1].scatter(
        timeline['date'],
        timeline['csc_max'],
        c=[ACTION_COLORS.get(action, '#7f7f7f') for action in timeline['action']],
        s=45,
        zorder=3,
    )
    axes[1].axvline(timeline.loc[index, 'date'], color='black', linestyle='--', linewidth=1)
    axes[1].legend(loc='upper left')
    axes[1].set_title('Replay timeline')
    axes[1].set_ylabel('Value')
    axes[1].set_xlabel('Date')
    plt.show()

    display(timeline.loc[[index], ['date', 'action', 'valid_fraction', 'csc_max', 'alert_pixels', 'note']])

widgets.interact(render_step, index=widgets.IntSlider(min=0, max=len(timeline) - 1, step=1, value=0));